# 4.2 Fitting the residual — noise, or a mechanism you forgot

04.1 fitted distributions to data. The most valuable thing to fit one to is **what a model
did not explain**: fit a model, subtract it, and ask the residual which family it belongs to.

- A **symmetric residual from a plausible family**, with no pattern over time, means the
  model extracted the structure it could. What is left is noise, and you should stop.
- A **residual with structure over time** — a step, a drift, a hump — is not a distribution
  problem at all. It is an unmodelled mechanism, and *when* it starts is usually the finding.

This notebook runs that loop on public Dutch COVID figures, deliberately not chat data: the
loop is the transferable part. It goes round twice. The first model is a straight line, and
on the window where its assumption holds the residual is noise. On the full window the same
line leaves a residual with a shape, the shape names the missing mechanism, and a second
model with that mechanism in it brings the residual back to noise. That is what "fitting
the residual" is for: it tells you whether you are done, and if not, what to add.

In [ ]:
import numpy as np
import pandas as pd

from goad_toolkit.analytics import DistributionFitter, FitResult, fit_table
from goad_toolkit.config import DataConfig, FileConfig
from goad_toolkit.dataprocessor import CovidDataProcessor
from goad_toolkit.models import linear_model, logistic, mse, train_model
from goad_toolkit.visualizer import (
    ComparePlot,
    ComparePlotDate,
    FitPlotSettings,
    PlotFits,
    PlotSettings,
    QQPlot,
    ResidualPlot,
)

## 4.2.1 The data, and the pipeline that makes it

Daily Dutch COVID figures, October 2020 to June 2021: positive tests and deaths per day.
`CovidDataProcessor` downloads the raw file once and runs a `Pipeline` over it — the same
kind of object lesson 1 built, with steps you can read:

1. `DiffValues` on `deaths` — the source reports the *cumulative* count; the difference is
   the number of deaths that day.
2. `ShiftValues(period=-14)` → `deaths_shifted` — a death follows the positive test by
   about two weeks, so the deaths recorded fourteen days *later* are moved back onto
   today's row, next to the tests that led to them. That is the lag a model between the two
   needs; without it, the line would be fitted to pairs that have nothing to do with each
   other.
3. `SelectDataRange` — the window.
4. `RollingAvg(window=7)` on both — reporting has a weekly cycle (fewer registrations at
   weekends) that is not part of the disease.
5. `ZScaler` → `*_zscore` — deaths are tens a day, tests are thousands. To see whether the
   two series have the same *shape*, put them on the same scale first.

In [ ]:
processor = CovidDataProcessor(FileConfig(), DataConfig())
data = processor.process()
print(f"{len(data)} days, {data.index.min().date()} to {data.index.max().date()}")
data[["positivetests", "deaths", "deaths_shifted", "deaths_shifted_zscore", "positivetests_zscore"]].head(3)

In [ ]:
fig, ax = ComparePlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="z-score",
                                   title="Positive tests and (lagged) deaths, on one scale")).plot(
    data=data, x="date", y1="positivetests_zscore", y2="deaths_shifted_zscore",
)

Through the autumn and winter the two curves track each other: when tests rise, deaths two
weeks later rise by a proportional amount. That is exactly the assumption a straight line
makes — a constant ratio of deaths to tests — and it is worth saying out loud before fitting,
because the second half of the plot already hints where it stops holding: from March, tests
climb again and deaths do not follow.

## 4.2.2 A model on the window where it holds

Vaccination in the Netherlands started on 6 January 2021. Fit the line on the days before
that, where the ratio looks constant: `deaths_shifted ≈ a · positivetests + b`, with goad's
four-function modelling kit — `linear_model` is the shape, `mse` is the loss, `train_model`
finds `a` and `b`. Then look at what the line did not explain.

In [ ]:
VACCINATION_START = "2021-01-06"
before = data.index < VACCINATION_START

x = data["positivetests"].to_numpy()
y = data["deaths_shifted"].to_numpy()

params_before = train_model(x[before], y[before], linear_model, mse, [0.01, 1.0],
                            bounds=[(0, 1.0), (0, None)])
print(f"deaths ≈ {params_before[0]:.4f} · tests + {params_before[1]:.1f}")

early = data[before].copy()
early["predicted"] = linear_model(x[before], params_before)
early["residual"] = early["deaths_shifted"] - early["predicted"]

fig, ax = ComparePlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="deaths",
                                   title="Before vaccination: a straight line in positive tests")).plot(
    data=early, x="date", y1="deaths_shifted", y2="predicted",
)

In [ ]:
fig, ax = ResidualPlot(PlotSettings(figsize=(11, 3.5), xlabel="date", ylabel="error",  # ty: ignore[invalid-argument-type]
                                    title="What the line did not explain, before vaccination")).plot(
    data=early, x="date", y="residual", date=VACCINATION_START, datelabel="vaccination starts",
    interval=2,
)

Waves of a week or two either side of zero — what a 7-day smoothing of noisy daily counts
leaves behind — and no drift across the window. Look honestly at the last two weeks, though:
the line falls short of the winter peak, a run of positive errors right up to the edge. Keep
that in mind; it is the first hint of the shape the full window will show. The first check
is otherwise clear, so on to the second: the distribution of the error itself. This is where
`DistributionFitter` earns its place in a modelling loop: **test the residual**. Fit every
continuous family and read the table for shape and for agreement, not for a single winner.

In [ ]:
fitter = DistributionFitter(seed=42)
fits_before = fitter.fit(early["residual"].to_numpy(), discrete=False)
fit_table(fits_before)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]]

In [ ]:
fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="error", ylabel="density",
                            title="Before vaccination: the residual, top three fits")).plot(
    data=early["residual"].to_numpy(), fit_results=fits_before,
    fitplotsettings=FitPlotSettings(bins=25, max_fits=3),
)

Centred on zero, and half a dozen families within a few log-likelihood points of each other
— beta, weibull, gamma, lognormal, skew-normal, normal — none of them rejected by KS. The one
family that *is* rejected, `uniform`, is rejected for the right reason: the errors cluster
near zero rather than spreading evenly. When the winner is not meaningfully ahead, the data
has not distinguished the families, and that is what noise looks like. Read it as a verdict
on the model: on this window, one ratio of deaths to tests is very nearly the whole story.

## 4.2.3 The same model on the full window

Now fit the same straight line on all 232 days. Nothing in it knows a vaccine exists.

In [ ]:
params_full = train_model(x, y, linear_model, mse, [0.01, 1.0], bounds=[(0, 1.0), (0, None)])
data["linear"] = linear_model(x, params_full)
data["residual_linear"] = data["deaths_shifted"] - data["linear"]
print(f"deaths ≈ {params_full[0]:.4f} · tests + {params_full[1]:.1f}   (mse {mse(y, data['linear']):.0f})")

fig, ax = ResidualPlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="error",
                                    title="The full window: a residual with a shape")).plot(
    data=data, x="date", y="residual_linear", date=VACCINATION_START,
    datelabel="vaccination starts", interval=1,
)

In [ ]:
fits_full = fitter.fit(data["residual_linear"].to_numpy(), discrete=False)
fit_table(fits_full)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]].head(4)

Two things changed, and they are the same thing seen twice. The residual plot has **a
shape**: positive through the winter wave, then from February a long negative run that
never comes back — the line predicts more deaths than happen, month after month. And the
fit table gives up: every family is rejected, and the family that "wins" is `uniform`,
which is what a fitter says when the errors are spread along a trend rather than scattered
around zero.

That is not a distribution problem. The residual is telling you what the model is missing,
and roughly when: **after vaccination, the same number of positive tests produces fewer
deaths.** A single fixed ratio has no way to say that. The residual just did.

## 4.2.4 Fix the model: the ratio turns

Write the missing mechanism down as a term. The ratio of deaths to tests was one number,
then it became a smaller one, and the change was not a step on a single day but a transition
over weeks as protection built up in the population. A **logistic** curve is the standard
shape for exactly that: flat, a turn, flat again. Two parameters decide it — `x0`, the day
the turn is halfway, and `k`, how steep it is (negative: down from 1 to 0).

In [ ]:
t = np.linspace(-10, 10, 200)
shapes = pd.DataFrame({"t": t, "k=1: up": logistic(t, k=1, x0=0), "k=-0.5: down, slower": logistic(t, k=-0.5, x0=0)})
fig, ax = ComparePlot(PlotSettings(figsize=(7, 3.2), xlabel="t", ylabel="", title="A logistic switch")).plot(  # ty: ignore[invalid-argument-type]
    data=shapes, x="t", y1="k=1: up", y2="k=-0.5: down, slower",
)

The model is the same straight line, **multiplied by the switch**:

    deaths_shifted ≈ (a · positivetests + b) · logistic(day, k, x0)

so the ratio runs at its old value while the switch is at 1, and decays as it turns. The
model function takes two inputs per day — the tests and the day number — stacked into one
array, and four parameters. Initial values come from what you already know: `a` and `b`
from the linear fit, `k` gently negative, `x0` a few weeks after vaccination started.

In [ ]:
day = np.arange(len(data)).astype(float)
X = np.stack([x, day], axis=1)


def covid_model(X: np.ndarray, params: list[float]) -> np.ndarray:
    """A ratio of deaths to tests that turns down along a logistic curve."""
    a, b, k, x0 = params
    return linear_model(X[:, 0], [a, b]) * logistic(X[:, 1], k=k, x0=x0)


vaccination_day = float(np.argmax(data.index >= VACCINATION_START))
initial = [params_full[0], params_full[1], -0.1, vaccination_day + 30]
params = train_model(X, y, covid_model, mse, initial,
                     bounds=[(0, 1.0), (0, None), (-1.0, 0), (0, len(data))])
a, b, k, x0 = params

data["turning"] = covid_model(X, params)
data["residual_turning"] = data["deaths_shifted"] - data["turning"]
print(f"a = {a:.4f}, b = {b:.1f}, k = {k:.3f}, x0 = day {x0:.0f} = {data.index[int(round(x0))].date()}")
print(f"mse: straight line {mse(y, data['linear']):.0f}  →  with the switch {mse(y, data['turning']):.0f}")

In [ ]:
fig, ax = ComparePlotDate(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="deaths",
                                       title="Deaths, and a model whose ratio turns")).plot(
    data=data, x="date", y1="deaths_shifted", y2="turning",
    date=VACCINATION_START, datelabel="vaccination starts",
)

In [ ]:
fig, ax = ResidualPlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="error",
                                    title="The residual, once the ratio is allowed to turn")).plot(
    data=data, x="date", y="residual_turning", date=VACCINATION_START,
    datelabel="vaccination starts", interval=1,
)

In [ ]:
fits_turning = fitter.fit(data["residual_turning"].to_numpy(), discrete=False)
print(fit_table(fits_turning)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]].head(4).to_string())

fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="error", ylabel="density",
                            title="Full window, with the switch: the residual, top three fits")).plot(
    data=data["residual_turning"].to_numpy(), fit_results=fits_turning,
    fitplotsettings=FitPlotSettings(bins=30, max_fits=3),
)

The long negative run is gone, the error is a seventh of what it was, and the residual is
back to what it looked like on the pre-vaccination window: symmetric, several families tied,
none rejected. The loop closed — not because the fit is perfect (the weekly waves are still
there, and a serious model of this would treat them), but because the residual no longer
carries a *mechanism* the model lacks.

Read the parameters as a finding, carefully. The switch is halfway around **mid-March 2021**,
about ten weeks after the first vaccinations, and by May the ratio has fallen to a fraction
of its winter value. That is what the data says about *when* the deaths-per-test ratio
changed. What it does not say is *why*: the model has a switch on the calendar, not a term
for vaccination, and the end of the winter wave, changes in who was being tested, and the
order in which age groups were vaccinated all sit on the same calendar. The model dates the
turn; the mechanism is a claim you would have to defend with something other than this fit.

## 4.2.5 Outliers: a point in the tail is a question, not a verdict

With a fitted family in hand, "is this point unusual?" has an honest answer: how much tail
probability does *the family you committed to* put beyond it. The standard z-score rule
answers a different question — how unusual would it be *if the data were normal* — and the
two disagree exactly when the family is not normal.

In [ ]:
residual = data["residual_turning"].to_numpy()
best = next(f for f in fits_turning if isinstance(f, FitResult) and f.best_likelihood)

z_flagged = np.abs((residual - residual.mean()) / residual.std()) > 3
tail = np.minimum(best.frozen_dist.cdf(residual), 1 - best.frozen_dist.cdf(residual))
tail_flagged = tail < 0.005  # roughly the two-sided z>3 threshold, but under the fitted family

print(f"z-score rule (assumes normal):        {z_flagged.sum()} day(s) flagged")
print(f"tail probability under {best.distribution:9s}: {tail_flagged.sum()} day(s) flagged")
print(f"the rules disagree on {int((z_flagged != tail_flagged).sum())} day(s)")
fig, ax = QQPlot(PlotSettings(title=f"residual vs. fitted {best.distribution}",
                              xlabel="theoretical quantile", ylabel="error")).plot(
    data=residual, distribution=best.frozen_dist,
)

Whichever way the two rules disagree, the fitted family's own tail probability is the more
defensible number: it is the distribution the rest of the analysis already committed to, not
a second assumption smuggled in through a rule of thumb. What to *do* with a flagged point —
a recording error to drop, a real rare day to keep, a regime the model has no term for, or a
wrong family rather than a wrong point — is a decision that needs knowledge of the process,
not more statistics.

That knowledge is yours, not the assistant's, which is why this is an MCP conversation
rather than a paragraph. Ask the `goad` server for its Distributions chapter and read §5.6,
"Outliers, properly" — five steps, and the trap the z-score rule walks into:

    goad_get_concept("Distributions")

Then take the flagged days from **your own chat** — the biggest residuals from 03.3's event
plot, or the days furthest into the tail of 04.1's messages-per-day fit — and go through the
five steps with the assistant, out loud. It can compute the tail probability; only you know
whether that Saturday was a birthday, an export glitch, or the week someone joined. Say
which, and say why. That is the answer to write down, not the number.

## Reflection

1. In your own words: why does `DistributionFitter` mark a *likelihood* winner and a *KS*
   winner separately, and what did their agreement (or not) tell you at each of the three
   fits in this notebook?
2. The residual in §4.2.3 was "rejected by every family". Explain why that is a statement
   about the *model*, not about the distributions — and name the mechanism it pointed at.
3. The switch dates the turn to mid-March. Write down two mechanisms other than vaccination
   that could produce the same turn, and what data would separate them.
4. For your own chat: pick one day 04.1's fit called unusual, and give the reason the
   z-score rule would have got it wrong there.